# Stage 02 — Watermark Removal (LaMa)
Detection JSON'dan watermark bolgelerini okur, LaMa inpainting ile siler.

Her watermark bolgesi ayri ayri kirpilip islenir (ROI-per-region).
Kucuk kirpma = hizli inference + yuksek kalite.

GPU runtime onerilir.

## 1 — Config + Kurulum

In [ ]:
import os, json, time, shutil, subprocess, gc
import cv2
import numpy as np
from pathlib import Path
from IPython.display import Video, display, HTML

if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

PIPELINE_ROOT = "/content/drive/MyDrive/pipeline"
JSON_PATH     = Path(PIPELINE_ROOT) / "watermark_detected" / "results.json"
OUTPUT_DIR    = Path(PIPELINE_ROOT) / "watermark_removed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WORK_DIR = Path("/content/wm_work")
WORK_DIR.mkdir(parents=True, exist_ok=True)

ROI_PADDING = 60  # her bolge etrafina ekstra piksel (LaMa icin context)
MASK_DILATION = 15  # watermark mask genisletme (piksel)
BLEND_RADIUS = 12  # sinir yumusatma (piksel)
SHARPEN_AMOUNT = 0.3  # keskinlestirme miktari (0=kapali, 0.3=hafif)
GRAIN_MATCH = True  # orijinal video grainini inpaint bolgesine ekle

# LaMa: numpy'a dokunmadan kur (Colab numpy 2.x ile calisiyor)
!pip install -q simple-lama-inpainting --no-deps

print(f"Input : {JSON_PATH}")
print(f"Output: {OUTPUT_DIR}")

## 2 — JSON Oku + Video Listesi

In [ ]:
with open(JSON_PATH, "r") as f:
    data = json.load(f)

videos = data["videos"]
wm_list    = {k: v for k, v in videos.items() if v["status"] == "WATERMARKED"}
clean_list = {k: v for k, v in videos.items() if v["status"] == "CLEAN"}

print(f"Toplam  : {len(videos)} video")
print(f"WATERMARKED : {len(wm_list)} (LaMa)")
print(f"CLEAN       : {len(clean_list)} (kopya)")

for name, info in wm_list.items():
    n = len(info['regions'])
    print(f"  {name} ({n} bolge)")

## 3 — LaMa Model + Helper Fonksiyonlar

In [ ]:
import torch
from simple_lama_inpainting import SimpleLama
from PIL import Image

lama_model = SimpleLama()
print(f"LaMa yuklendi. GPU: {torch.cuda.is_available()}")


def prepare_regions(regions, frame_h, frame_w, padding=ROI_PADDING):
    """Her bolge icin ROI, mask ve blend mask onceden hesaplar."""
    prepared = []
    for r in regions:
        x1, y1, x2, y2 = r["bbox"]
        rx1 = max(0, x1 - padding)
        ry1 = max(0, y1 - padding)
        rx2 = min(frame_w, x2 + padding)
        ry2 = min(frame_h, y2 + padding)
        roi_h = ry2 - ry1
        roi_w = rx2 - rx1

        # Inpaint mask (watermark bolgesi)
        mask = np.zeros((roi_h, roi_w), dtype=np.uint8)
        mx1, my1 = x1 - rx1, y1 - ry1
        mx2, my2 = x2 - rx1, y2 - ry1
        mask[my1:my2, mx1:mx2] = 255
        kernel = np.ones((MASK_DILATION, MASK_DILATION), np.uint8)
        mask = cv2.dilate(mask, kernel, iterations=1)
        mask_pil = Image.fromarray(mask).convert("L")

        # Blend mask (sinir yumusatma icin)
        # mask bolgesi=1, kenarlar gradient, dis=0
        blend = (mask.astype(np.float32) / 255.0)
        blend = cv2.GaussianBlur(blend, (0, 0), BLEND_RADIUS)
        # 3 kanala genislet (BGR blend icin)
        blend_3ch = np.stack([blend]*3, axis=-1)

        prepared.append({
            "roi": (rx1, ry1, rx2, ry2),
            "mask_pil": mask_pil,
            "blend": blend_3ch,
            "size": f"{roi_w}x{roi_h}"
        })
    return prepared


def inpaint_frame(frame, prepared_regions):
    """Tek frame'deki tum watermark bolgelerini LaMa ile siler.
    Grain ekleme + keskinlestirme + sinir yumusatma."""
    for pr in prepared_regions:
        rx1, ry1, rx2, ry2 = pr["roi"]
        roi_h, roi_w = ry2 - ry1, rx2 - rx1
        roi_orig = frame[ry1:ry2, rx1:rx2].astype(np.float32)

        # LaMa inpaint
        roi_pil = Image.fromarray(cv2.cvtColor(frame[ry1:ry2, rx1:rx2], cv2.COLOR_BGR2RGB))
        result_pil = lama_model(roi_pil, pr["mask_pil"])
        roi_lama = cv2.cvtColor(np.array(result_pil), cv2.COLOR_RGB2BGR)[:roi_h, :roi_w].astype(np.float32)

        # Keskinlestirme (unsharp mask)
        if SHARPEN_AMOUNT > 0:
            blurred = cv2.GaussianBlur(roi_lama, (0, 0), 1.5)
            roi_lama = cv2.addWeighted(roi_lama, 1.0 + SHARPEN_AMOUNT, blurred, -SHARPEN_AMOUNT, 0)

        # Grain ekleme (orijinal noise seviyesini tasi)
        if GRAIN_MATCH:
            # Orijinalin grainini olc (mask disi bolgeden)
            noise = roi_orig - cv2.GaussianBlur(roi_orig, (0, 0), 2.0)
            sigma = np.std(noise)
            # Ayni seviyede grain uret ve LaMa sonucuna ekle
            grain = np.random.normal(0, sigma, roi_lama.shape).astype(np.float32)
            roi_lama = np.clip(roi_lama + grain, 0, 255)

        # Blend: lama * blend + orijinal * (1-blend)
        blend = pr["blend"]
        blended = np.clip(roi_lama * blend + roi_orig * (1.0 - blend), 0, 255).astype(np.uint8)
        frame[ry1:ry2, rx1:rx2] = blended
    return frame


def remove_watermark(video_path, regions, output_path):
    """Video'daki watermarklari LaMa ile siler."""
    t0 = time.time()
    local_in = str(WORK_DIR / "input.mp4")
    local_raw = str(WORK_DIR / "raw.mp4")
    local_out = str(WORK_DIR / "out.mp4")

    try:
        shutil.copy2(video_path, local_in)

        cap = cv2.VideoCapture(local_in)
        fps = cap.get(cv2.CAP_PROP_FPS)
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # Bolgeleri onceden hesapla
        prepped = prepare_regions(regions, h, w)
        print(f"    Video: {w}x{h}, {total} frame, {fps:.0f}fps")
        for i, pr in enumerate(prepped):
            print(f"    Bolge {i+1}: ROI {pr['size']}")

        out = cv2.VideoWriter(local_raw, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

        count = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            out.write(inpaint_frame(frame, prepped))
            count += 1
            if count % 50 == 0:
                print(f"    {count}/{total} frame")

        cap.release()
        out.release()

        # h264 + ses koru
        cmd = [
            "ffmpeg", "-y", "-i", local_raw, "-i", local_in,
            "-map", "0:v", "-map", "1:a?",
            "-c:v", "libx264", "-preset", "ultrafast",
            "-c:a", "copy",
            local_out, "-loglevel", "error"
        ]
        subprocess.run(cmd)

        shutil.copy2(local_out, output_path)
        elapsed = time.time() - t0
        print(f"    Tamamlandi: {elapsed:.1f}sn ({count} frame, {count/elapsed:.0f} fps)")
        return elapsed

    except Exception as e:
        print(f"    HATA: {e}")
        return -1

    finally:
        for f in [local_in, local_raw, local_out]:
            if os.path.exists(f):
                os.remove(f)


print("Helper fonksiyonlar hazir.")

## 4 — Ana Dongu

In [ ]:
t_start = time.time()
results_log = []

total_videos = len(wm_list) + len(clean_list)
processed = 0

for name, info in wm_list.items():
    processed += 1
    src = info["source_path"]
    dst = str(OUTPUT_DIR / name)
    n_regions = len(info["regions"])

    print(f"\n[{processed}/{total_videos}] LAMA: {name} ({n_regions} bolge)")

    elapsed = remove_watermark(src, info["regions"], dst)
    action = "LAMA" if elapsed > 0 else "HATA"
    results_log.append({
        "video": name, "action": action,
        "regions": n_regions, "seconds": round(max(elapsed, 0), 1)
    })

for name, info in clean_list.items():
    processed += 1
    src = info["source_path"]
    dst = str(OUTPUT_DIR / name)

    print(f"\n[{processed}/{total_videos}] KOPYALA: {name}")
    shutil.copy2(src, dst)
    results_log.append({
        "video": name, "action": "COPY",
        "regions": 0, "seconds": 0
    })

total_elapsed = time.time() - t_start
print(f"\nTamamlandi: {total_elapsed:.1f}sn")

## 5 — GPU Temizle

In [ ]:
del lama_model
gc.collect()
torch.cuda.empty_cache()
print("GPU bellek serbest birakildi.")

## 6 — Ozet

In [ ]:
print(f"{'Video':<45} {'Islem':<8} {'Bolge':>5} {'Sure':>8}")
print("-" * 70)
for r in results_log:
    sure = f"{r['seconds']}sn" if r['seconds'] > 0 else "-"
    print(f"{r['video']:<45} {r['action']:<8} {r['regions']:>5} {sure:>8}")
print("-" * 70)

n_lama = sum(1 for r in results_log if r['action'] == 'LAMA')
n_copy = sum(1 for r in results_log if r['action'] == 'COPY')
n_err  = sum(1 for r in results_log if r['action'] == 'HATA')
print(f"\n{n_lama} LaMa | {n_copy} kopya | {n_err} hata")
print(f"Toplam sure: {total_elapsed:.1f}sn")
print(f"\nCikti: {OUTPUT_DIR}")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name} ({f.stat().st_size / (1024*1024):.1f} MB)")

## 7 — Video Preview

In [ ]:
output_videos = sorted([f for f in OUTPUT_DIR.iterdir() if f.suffix.lower() in ('.mp4','.mov','.avi','.mkv','.webm')])

print("Mevcut videolar:")
for i, f in enumerate(output_videos):
    tag = "[LAMA]" if f.name in wm_list else "[CLEAN]"
    print(f"  {i}: {f.name} {tag}")

def preview(n):
    if n < 0 or n >= len(output_videos):
        print(f"0-{len(output_videos)-1} arasi gir.")
        return
    f = output_videos[n]
    local = str(WORK_DIR / f.name)
    shutil.copy2(str(f), local)
    tag = "LAMA" if f.name in wm_list else "CLEAN"
    display(HTML(f"<h3>[{n}] {f.name} — {tag}</h3>"))
    display(Video(local, embed=True, width=720))
    os.remove(local)

def preview_all():
    for i in range(len(output_videos)):
        preview(i)

preview_all()